# Soccer tactical raw aggregate features

This notebook generates `features.csv` with 20 rows if the 10-match SkillCorner sample is available: one row per match-team. It uses only raw counts, durations, and distance sums.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# -----------------------------
# Locate dynamic event CSV files
# -----------------------------
def find_dynamic_event_files():
    candidates = [
        Path("/kaggle/input").rglob("opendata/data/matches/*/*_dynamic_events.csv"),
        Path("/kaggle/working").rglob("opendata/data/matches/*/*_dynamic_events.csv"),
        Path(".").rglob("opendata/data/matches/*/*_dynamic_events.csv"),
        Path("/kaggle/input").rglob("*_dynamic_events.csv"),
        Path("/kaggle/working").rglob("*_dynamic_events.csv"),
        Path(".").rglob("*_dynamic_events.csv"),
    ]
    for pattern in candidates:
        files = sorted(list(pattern))
        if files:
            return files
    raise FileNotFoundError("No *_dynamic_events.csv files found.")

dynamic_files = find_dynamic_event_files()
print("Dynamic event files found:", len(dynamic_files))

# -----------------------------
# Safe helpers
# -----------------------------
def safe_num(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0, index=df.index)

def safe_str(df, col):
    if col in df.columns:
        return df[col].astype(str).str.lower().replace("nan", "")
    return pd.Series("", index=df.index)

def safe_bool(df, col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    s = df[col]
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0)
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])

def count_bool(cond):
    return int(pd.Series(cond).fillna(False).sum())

def count_eq(s, value):
    return int((s.fillna("") == value).sum())

def sum_col(df, col):
    return float(safe_num(df, col).sum())

def sum_positive(series):
    return float(pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0).sum())

# -----------------------------
# Feature creation
# -----------------------------
SELECTED_FEATURES = [
    "pp_build_up_duration_sum",
    "pp_create_duration_sum",
    "pp_finish_duration_sum",
    "pp_quick_break_duration_sum",
    "pp_transition_duration_sum",
    "pp_direct_duration_sum",
    "pp_set_play_duration_sum",
    "pp_chaotic_duration_sum",

    "pp_count",
    "pp_duration_sum",
    "pp_distance_sum",
    "pp_forward_count",
    "pp_backward_count",
    "pp_sideway_count",
    "pp_forward_distance_sum",
    "pp_backward_distance_sum",
    "pp_lateral_distance_sum",
    "pp_end_attacking_third_count",
    "pp_end_penalty_area_count",
    "pp_shot_end_count",
    "pp_loss_end_count",
    "pp_lead_to_shot_count",

    "pass_short_count",
    "pass_long_count",
    "pass_forward_count",
    "pass_backward_count",
    "pass_sideway_count",

    "po_count",
    "po_ahead_start_count",
    "po_wide_start_count",
    "po_halfspace_start_count",
    "po_center_start_count",
    "po_end_attacking_third_count",
    "po_end_penalty_area_count",
    "po_short_distance_count",
    "po_long_distance_count",
    "po_dangerous_count",

    "obr_count",
    "obr_behind_count",
    "obr_support_count",
    "obr_overlap_underlap_count",
    "obr_cross_receiver_count",
    "obr_break_defensive_line_count",
    "obr_push_defensive_line_count",
    "obr_give_and_go_count",

    "obe_pressing_count",
    "obe_counter_press_count",

    "passes_before_loss_total",
    "loss_after_0_pass_count",
    "loss_after_3_to_5_pass_count",
]

def feature_row_for_team(df, team_id):
    row = {}
    team = df[pd.to_numeric(df["team_id"], errors="coerce") == team_id].copy()

    et = safe_str(team, "event_type")
    pp = team[et == "player_possession"].copy()
    po = team[et == "passing_option"].copy()
    obr = team[et == "off_ball_run"].copy()
    obe = team[et == "on_ball_engagement"].copy()

    pp_phase = safe_str(pp, "team_in_possession_phase_type")
    pp_traj = safe_str(pp, "trajectory_direction")
    pp_end_type = safe_str(pp, "end_type")
    pp_end_third = safe_str(pp, "third_end")
    pp_start_channel = safe_str(pp, "channel_start")
    pp_pass_range = safe_str(pp, "pass_range")
    pp_pass_dir = safe_str(pp, "pass_direction")

    x_delta = safe_num(pp, "x_end") - safe_num(pp, "x_start")
    y_delta = safe_num(pp, "y_end") - safe_num(pp, "y_start")

    # Phase duration sums using player-possession durations.
    for ph in ["build_up", "create", "finish", "quick_break", "transition", "direct", "set_play", "chaotic"]:
        mask = pp_phase == ph
        row[f"pp_{ph}_duration_sum"] = float(safe_num(pp, "duration")[mask].sum())

    # Possession movement and outcomes.
    row["pp_count"] = int(len(pp))
    row["pp_duration_sum"] = sum_col(pp, "duration")
    row["pp_distance_sum"] = sum_col(pp, "distance_covered")
    row["pp_forward_count"] = count_eq(pp_traj, "forward")
    row["pp_backward_count"] = count_eq(pp_traj, "backward")
    row["pp_sideway_count"] = int(pp_traj.isin(["sideway_left", "sideway_right"]).sum())
    row["pp_forward_distance_sum"] = sum_positive(x_delta)
    row["pp_backward_distance_sum"] = sum_positive(-x_delta)
    row["pp_lateral_distance_sum"] = float(y_delta.abs().sum())
    row["pp_end_attacking_third_count"] = count_eq(pp_end_third, "attacking_third")
    row["pp_end_penalty_area_count"] = count_bool(safe_bool(pp, "penalty_area_end"))
    row["pp_shot_end_count"] = count_eq(pp_end_type, "shot")
    row["pp_loss_end_count"] = count_eq(pp_end_type, "possession_loss")
    row["pp_lead_to_shot_count"] = count_bool(safe_bool(pp, "lead_to_shot"))

    # Pass style.
    pass_mask = pp_end_type == "pass"
    row["pass_short_count"] = int((pass_mask & (pp_pass_range == "short")).sum())
    row["pass_long_count"] = int((pass_mask & (pp_pass_range == "long")).sum())
    row["pass_forward_count"] = int((pass_mask & (pp_pass_dir == "forward")).sum())
    row["pass_backward_count"] = int((pass_mask & (pp_pass_dir == "backward")).sum())
    row["pass_sideway_count"] = int((pass_mask & pp_pass_dir.isin(["sideway_left", "sideway_right"])).sum())

    # Passing option structure.
    po_loc_start = safe_str(po, "location_to_player_in_possession_start")
    po_channel_start = safe_str(po, "channel_start")
    po_end_third = safe_str(po, "third_end")
    po_dist_range = safe_str(po, "interplayer_distance_range")
    row["po_count"] = int(len(po))
    row["po_ahead_start_count"] = count_eq(po_loc_start, "ahead")
    row["po_wide_start_count"] = int(po_channel_start.isin(["wide_left", "wide_right"]).sum())
    row["po_halfspace_start_count"] = int(po_channel_start.isin(["half_space_left", "half_space_right"]).sum())
    row["po_center_start_count"] = count_eq(po_channel_start, "center")
    row["po_end_attacking_third_count"] = count_eq(po_end_third, "attacking_third")
    row["po_end_penalty_area_count"] = count_bool(safe_bool(po, "penalty_area_end"))
    row["po_short_distance_count"] = count_eq(po_dist_range, "short")
    row["po_long_distance_count"] = count_eq(po_dist_range, "long")
    row["po_dangerous_count"] = count_bool(safe_bool(po, "dangerous"))

    # Off-ball movement / combination movement.
    obr_sub = safe_str(obr, "event_subtype")
    row["obr_count"] = int(len(obr))
    row["obr_behind_count"] = count_eq(obr_sub, "behind")
    row["obr_support_count"] = count_eq(obr_sub, "support")
    row["obr_overlap_underlap_count"] = int(obr_sub.isin(["overlap", "underlap"]).sum())
    row["obr_cross_receiver_count"] = count_eq(obr_sub, "cross_receiver")
    row["obr_break_defensive_line_count"] = count_bool(safe_bool(obr, "break_defensive_line"))
    row["obr_push_defensive_line_count"] = count_bool(safe_bool(obr, "push_defensive_line"))
    row["obr_give_and_go_count"] = count_bool(safe_bool(obr, "give_and_go"))

    # Defensive pressure.
    obe_sub = safe_str(obe, "event_subtype")
    row["obe_pressing_count"] = count_eq(obe_sub, "pressing")
    row["obe_counter_press_count"] = count_eq(obe_sub, "counter_press")

    # Passes before loss, using phase_index as the team-possession sequence proxy.
    row["passes_before_loss_total"] = 0
    row["loss_after_0_pass_count"] = 0
    row["loss_after_3_to_5_pass_count"] = 0

    if "phase_index" in pp.columns:
        sort_cols = [c for c in ["period", "frame_start", "time_start", "index"] if c in pp.columns]
        pp_sorted = pp.sort_values(sort_cols) if sort_cols else pp.copy()

        for _, grp in pp_sorted.groupby("phase_index", dropna=True):
            pass_count = 0
            for _, r in grp.iterrows():
                end_type = str(r.get("end_type", "")).lower()
                if end_type == "pass":
                    pass_count += 1
                elif end_type == "possession_loss":
                    row["passes_before_loss_total"] += pass_count
                    if pass_count == 0:
                        row["loss_after_0_pass_count"] += 1
                    elif 3 <= pass_count <= 5:
                        row["loss_after_3_to_5_pass_count"] += 1
                    break

    return row

# -----------------------------
# Run all files
# -----------------------------
rows = []
for path in dynamic_files:
    df = pd.read_csv(path, low_memory=False)

    if "match_id" in df.columns and not df["match_id"].dropna().empty:
        match_id = int(pd.to_numeric(df["match_id"], errors="coerce").dropna().iloc[0])
    else:
        match_id = int(re.search(r"(\d+)_dynamic_events\.csv", path.name).group(1))

    team_ids = sorted(pd.to_numeric(df["team_id"], errors="coerce").dropna().astype(int).unique().tolist())

    for team_id in team_ids:
        row = {"match_id": match_id, "team_id": team_id}
        row.update(feature_row_for_team(df, team_id))
        rows.append(row)

features = pd.DataFrame(rows)

# Keep exactly the required ID columns + selected raw aggregate features.
features = features[["match_id", "team_id"] + SELECTED_FEATURES]
features = features.sort_values(["match_id", "team_id"]).reset_index(drop=True).fillna(0)

print("features.csv shape:", features.shape)
display(features.head())

features.to_csv("features.csv", index=False)
print("Saved features.csv")